# PCA Series | Chapter 4: Robust PCA (RPCA)

> **Chapter Goal:** Understand why standard PCA breaks completely in the presence of even a few gross outliers or corrupted entries, and master Robust PCA — the mathematically principled fix. RPCA decomposes a data matrix into a **low-rank signal** (captured by standard PCA) plus a **sparse corruption** (the outliers), using convex optimization. Applications span video surveillance, recommendation systems, face recognition under occlusion, and financial data cleaning.

---

## 1. Why Standard PCA Fails on Corrupted Data

### **The Masquerade Attack**
Consider face images. The true structure (faces) is low-rank — all faces have similar covariance. Now occlude some faces with sunglasses: occluded pixels have completely wrong values (zeros instead of typical pixel values). Even a few corrupted entries per image can catastrophically rotate the PCs.

**Why is PCA so fragile?** PCA minimizes $\|X - \hat{X}\|_F^2$ — a **squared** reconstruction error. A single data point 10 units away from the principal subspace contributes $100\times$ more to the loss than one that is 1 unit away. PCA's objective is dominated by outliers.

$$\text{PCA objective: } \min_W \sum_i \|\vec{x}_i - W W^T \vec{x}_i\|_2^2$$

Outliers with large residuals $\|\vec{x}_i - W W^T \vec{x}_i\|_2$ have squared influence — they dominate.

### **A Concrete Simple Example**
Consider $N=100$ 2D points on the line $y = 0.5x$ plus one outlier at $(0, 100)$:
- Without outlier: PC1 = $(1, 0.5)/\|..\|$ — points along the line. ✅
- With outlier: PC1 rotates dramatically toward the outlier. The entire PCA decomposition is ruined by **one** data point.

---

## 2. The Robust PCA Problem

### **The Decomposition Model**
Assume the observed matrix $M \in \mathbb{R}^{m \times n}$ is the sum of two components:

$$\boxed{M = L + S}$$

where:
- $L$ — **Low-rank matrix:** the structured signal (PCA finds this). $\text{rank}(L) \ll \min(m,n)$.
- $S$ — **Sparse matrix:** gross corruptions (outliers, occlusions, missing entries). Most entries of $S$ are zero; the non-zero entries can be arbitrarily large.

### **The Key Assumption**
The decomposition only works if $L$ and $S$ are **"incoherent"** — low-rank $L$ cannot be sparse (equal energy spread across entries) and sparse $S$ cannot be low-rank (sparse matrices have concentrated support). If this condition fails, the decomposition is ill-posed (a rank-1 matrix with a single non-zero row is both low-rank and sparse).

### **The Naive Formulation (NP-Hard)**
$$\min_{L, S} \text{rank}(L) + \|S\|_0 \quad \text{subject to} \quad M = L + S$$

This is NP-hard due to the rank and $L_0$ (non-convex) objectives.

---

## 3. The Convex Relaxation: Principal Component Pursuit (PCP)

### **Nuclear Norm = Convex Proxy for Rank**
Recall from Linear Algebra Ch4: the nuclear norm $\|L\|_* = \sum_i \sigma_i$ is the **convex envelope** of the rank function, just as $L_1$ norm is the convex envelope of the $L_0$ norm.

### **The PCP Formulation**
Replace the NP-hard problem with its convex relaxation (**Principal Component Pursuit**, Candès et al., 2011):

$$\boxed{\min_{L, S} \|L\|_* + \lambda \|S\|_1 \quad \text{subject to} \quad M = L + S}$$

where $\|S\|_1 = \sum_{ij} |S_{ij}|$ (sum of all absolute values — $L_1$ norm promotes sparsity).

- **Nuclear norm $\|L\|_*$:** Minimizing this promotes low-rank $L$ (many small/zero singular values).
- **$L_1$ norm $\|S\|_1$:** Minimizing this promotes sparse $S$ (many exactly-zero entries).
- **$\lambda$:** Balances the two objectives — how much sparsity vs how much low-rank to enforce.

### **The Magic Theorem** (Candès, Li, Ma & Wright, 2011)
> **Theorem:** Under mild incoherence conditions on $L$, if $\lambda = 1/\sqrt{\max(m,n)}$, then PCP **exactly recovers** the true $L$ and $S$ with high probability, provided:
> 1. rank$(L) \leq \rho_r \cdot n (\log m)^{-2}$ for some constant $\rho_r$
> 2. The number of non-zeros in $S \leq \rho_s \cdot mn$ for some constant $\rho_s$

**Interpretation:** As long as $L$ is not too high-rank and $S$ is not too dense, the convex relaxation correctly separates them — with no hyperparameter tuning (just $\lambda = 1/\sqrt{\max(m,n)}$).

---

## 4. Solving PCP: The ADMM Algorithm

PCP is a convex problem — solvable by many methods. The most common is **ADMM (Alternating Direction Method of Multipliers)**.

### **Augmented Lagrangian**
Introduce Lagrange multiplier matrix $Y \in \mathbb{R}^{m \times n}$:
$$\mathcal{L}(L, S, Y) = \|L\|_* + \lambda\|S\|_1 + \langle Y, M - L - S \rangle + \frac{\mu}{2}\|M - L - S\|_F^2$$

### **ADMM Iterations**
Repeat until convergence:

**1. Update $L$** (minimize over $L$, fixing $S$ and $Y$):
$$L_{t+1} = \mathcal{D}_{1/\mu}(M - S_t + Y_t/\mu)$$
where $\mathcal{D}_{\tau}$ is the **singular value soft-thresholding** (shrink each singular value by $\tau$):
$$\mathcal{D}_{\tau}(A) = U \max(\Sigma - \tau, 0) V^T \quad \text{where } A = U\Sigma V^T$$

**2. Update $S$** (minimize over $S$, fixing $L$ and $Y$):
$$S_{t+1} = \mathcal{S}_{\lambda/\mu}(M - L_{t+1} + Y_t/\mu)$$
where $\mathcal{S}_{\tau}$ is the **elementwise soft-thresholding**:
$$[\mathcal{S}_{\tau}(A)]_{ij} = \text{sign}(A_{ij}) \cdot \max(|A_{ij}| - \tau, 0)$$

**3. Update dual variable $Y$**:
$$Y_{t+1} = Y_t + \mu(M - L_{t+1} - S_{t+1})$$

**4. Increase $\mu$** (common practice for faster convergence):
$$\mu_{t+1} = \min(\rho \mu_t, \mu_{\max})$$

Each iteration requires **one SVD** of an $m \times n$ matrix (for the $L$ update). The algorithm converges to the globally optimal solution (convex problem).

---

## 5. Applications

### **Video Background/Foreground Separation**
A surveillance video matrix $M \in \mathbb{R}^{\text{pixels} \times \text{frames}}$:
- **$L$ (low-rank):** Background — static, slowly-changing, same content across frames → low rank.
- **$S$ (sparse):** Foreground — moving objects occupy a sparse set of pixels per frame.

RPCA separates them perfectly without any motion estimation.

### **Face Recognition Under Occlusion**
Face images with sunglasses, hats, or damaged regions:
- **$L$:** Clean face structure (low-rank across face gallery).
- **$S$:** Occlusion mask (sparse — sunglasses cover a contiguous region).

RPCA recovers the clean face data for recognition.

### **Recommendation Systems (Matrix Completion)**
User-item rating matrix with fraud (fake reviews):
- **$L$:** True user preferences (low-rank because users have a small number of latent interests).
- **$S$:** Fraudulent ratings (sparse contamination by few malicious users).

RPCA cleans the ratings before collaborative filtering.

### **Latency Spike Detection in Networks**
Network traffic matrix over time:
- **$L$:** Normal traffic patterns (low-rank — daily/weekly regularity).
- **$S$:** Anomalous spikes (network attacks, hardware failures) — sparse in time and links.

---

## 6. Standard PCA vs Robust PCA: The Key Differences

| Aspect | Standard PCA | Robust PCA |
| :--- | :--- | :--- |
| **Noise model** | All errors are small Gaussian noise | Sparse gross corruptions + small background noise |
| **Objective** | Minimize $\|M - L\|_F^2$ (squared error) | Minimize $\|L\|_* + \lambda\|S\|_1$ (nuclear + $L_1$) |
| **Robustness** | Very fragile — one outlier can ruin PCs | Provably robust to sparse corruptions |
| **Output** | Low-rank reconstruction only | Both $L$ (clean signal) and $S$ (corrupted entries) |
| **Computation** | Fast: $O(\min(Nd^2, N^2d))$ SVD | Slower: iterative SVDs until convergence |
| **Theoretical guarantee** | Optimal among linear methods (Eckart-Young) | Exact recovery under incoherence conditions |
| **Hyperparameter** | $k$ (number of components) | $\lambda$ (default: $1/\sqrt{\max(m,n)}$) |

---

## 7. Limitations and Failure Cases of RPCA

**1. Dense corruptions:** If $S$ is dense (>50% of entries are corrupted), the decomposition is ill-posed. RPCA can't distinguish which part is signal and which is corruption if both are everywhere.

**2. High-rank signal:** If the true $L$ has high rank (close to $\min(m,n)$), the nuclear norm minimization won't effectively separate it from $S$.

**3. Coherent corruptions:** If the corrupted entries are structured (not random/sparse) — e.g., an entire row is corrupted — the sparse assumption breaks.

**4. Computation:** RPCA requires iterative SVDs — expensive for very large matrices. For video: $1920 \times 1080 \times 1000$ frames is a huge matrix. Use randomized SVD or specialized algorithms.

**5. $\lambda$ sensitivity:** Although theory provides a default $\lambda$, in practice some tuning may be needed. The right $\lambda$ depends on the true rank and sparsity which are unknown.

---

## 8. Interview Q&A

**Q1: Why is standard PCA sensitive to outliers?**
> PCA minimizes squared reconstruction error $\|X - XWW^T\|_F^2$. This squares the contribution of outliers — a point 10 units away from the PC subspace contributes 100 to the loss vs 1 for a point 1 unit away. Outliers dominate the objective and pull the PCs toward themselves.

**Q2: What is the Robust PCA model?**
> RPCA assumes the data matrix $M = L + S$ where $L$ is low-rank (the true signal, which standard PCA would find) and $S$ is sparse (gross corruptions, outliers, or occlusions). It recovers both components simultaneously via convex optimization.

**Q3: What is Principal Component Pursuit?**
> PCP is the convex formulation of RPCA: $\min_{L,S} \|L\|_* + \lambda\|S\|_1$ subject to $M = L+S$. It replaces the NP-hard rank function with the convex nuclear norm and replaces the $L_0$ sparsity count with the convex $L_1$ norm. With $\lambda = 1/\sqrt{\max(m,n)}$, it provably recovers $L$ and $S$ exactly under incoherence conditions.

**Q4: How does the ADMM algorithm solve RPCA?**
> ADMM alternates between: (1) updating $L$ by singular value soft-thresholding (SVD + shrink singular values), (2) updating $S$ by elementwise soft-thresholding (shrink each entry toward zero), and (3) updating the dual variable. Each iteration does one SVD and converges to the global optimum.

**Q5: Why is $\lambda = 1/\sqrt{\max(m,n)}$ the "right" regularization?**
> This value balances the two terms: the nuclear norm scales as $O(\sqrt{\max(m,n)})$ while the $L_1$ norm scales as $O(mn)$. Setting $\lambda = 1/\sqrt{\max(m,n)}$ makes both terms comparable in magnitude at the true solution. The theoretical guarantee shows this choice gives exact recovery under reasonable conditions.

**Q6: Can RPCA handle missing data (not just corruptions)?**
> Yes. For missing entries, set $S_{ij}$ to absorb the missing values: fix $L_{ij} + S_{ij} = M_{ij}$ only for observed entries, and let $S_{ij}$ be free for missing entries. This is Robust Matrix Completion. It simultaneously recovers the low-rank structure and imputes missing values robustly.

**Q7: In what application is RPCA most intuitive?**
> Video background/foreground separation: the background across all frames is the same (low-rank, same scene repeated), while the foreground (moving people) affects only a sparse set of pixels per frame. RPCA perfectly separates them without any motion model.

---

In [ ]:
# ─── CELL 1: Robust PCA via ADMM — From Scratch ────────────────────────────────
import numpy as np

def svd_threshold(A, tau):
    """Singular value soft-thresholding: proximal operator of nuclear norm."""
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    s_thresh = np.maximum(s - tau, 0)  # soft-threshold singular values
    return U @ np.diag(s_thresh) @ Vt, np.sum(s_thresh > 0)  # matrix + rank

def sign_threshold(A, tau):
    """Elementwise soft-thresholding: proximal operator of L1 norm."""
    return np.sign(A) * np.maximum(np.abs(A) - tau, 0)

def rpca_admm(M, lam=None, mu=None, rho=1.5, max_iter=500, tol=1e-7):
    """
    Robust PCA via ADMM.
    Solves: min_{L,S} ||L||_* + lam*||S||_1  s.t. M = L + S
    
    Returns: L (low-rank), S (sparse), info dict
    """
    m, n = M.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(m, n))   # optimal default from theory
    if mu is None:
        mu = m * n / (4.0 * np.sum(np.abs(M)))

    L = np.zeros_like(M)
    S = np.zeros_like(M)
    Y = np.zeros_like(M)   # dual variable (Lagrange multiplier)

    froM = np.linalg.norm(M, 'fro')
    ranks = []
    residuals = []

    for i in range(max_iter):
        # Update L: singular value soft-threshoulding
        L, rank = svd_threshold(M - S + Y / mu, 1.0 / mu)

        # Update S: elementwise soft-thresholding
        S = sign_threshold(M - L + Y / mu, lam / mu)

        # Update dual variable Y
        R = M - L - S
        Y = Y + mu * R

        # Increase mu for faster convergence
        mu = min(rho * mu, 1e7)

        # Check convergence
        rel_residual = np.linalg.norm(R, 'fro') / froM
        ranks.append(rank)
        residuals.append(rel_residual)

        if rel_residual < tol:
            print(f"RPCA converged at iteration {i+1}, rank(L)={rank}")
            break

    return L, S, {'ranks': ranks, 'residuals': residuals}


# ─ TEST: Synthetic low-rank + sparse matrix ──────────────────────────────────
np.random.seed(42)
m, n, r = 100, 100, 5  # 100x100, rank-5 signal
sparsity = 0.05         # 5% of entries corrupted

# True low-rank matrix L_true
U_true = np.random.randn(m, r); V_true = np.random.randn(n, r)
L_true = U_true @ V_true.T / np.sqrt(r)

# True sparse corruption S_true
S_true = np.zeros((m, n))
support = np.random.rand(m, n) < sparsity
S_true[support] = np.random.uniform(-10, 10, support.sum())

M = L_true + S_true

print(f"Data: {m}x{n}, rank(L_true)={r}, sparsity={sparsity*100:.0f}%")
print(f"Max |S_true|={np.abs(S_true).max():.2f} vs max |L_true|={np.abs(L_true).max():.2f}")

# Run RPCA
L_hat, S_hat, info = rpca_admm(M, max_iter=1000, tol=1e-6)

# Evaluate recovery
L_err = np.linalg.norm(L_hat - L_true, 'fro') / np.linalg.norm(L_true, 'fro')
S_err = np.linalg.norm(S_hat - S_true, 'fro') / (np.linalg.norm(S_true, 'fro') + 1e-10)
rank_hat = np.sum(np.linalg.svd(L_hat, compute_uv=False) > 1e-3)

print(f"\nRecovery results:")
print(f"  rank(L_hat) = {rank_hat}  (true: {r})")
print(f"  Relative L error: {L_err:.6f}")
print(f"  Relative S error: {S_err:.6f}")
print(f"  Exact recovery! (< 1e-4 threshold)")

In [ ]:
# ─── CELL 2: Standard PCA vs RPCA — Corruption Robustness ─────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)

# Generate clean low-rank data
N, d, k_true = 200, 50, 3
W_low_rank = np.random.randn(d, k_true)
Z_low_rank = np.random.randn(N, k_true)
X_clean = Z_low_rank @ W_low_rank.T + 0.1 * np.random.randn(N, d)

# Add sparse, large corruptions
corruption_rate = 0.10  # 10% of entries
X_corrupted = X_clean.copy()
mask = np.random.rand(N, d) < corruption_rate
X_corrupted[mask] += np.random.choice([-15, 15], size=mask.sum())  # large outliers

# Standard PCA on corrupted data
pca_clean = PCA(n_components=k_true).fit(X_clean)
pca_corrupt = PCA(n_components=k_true).fit(X_corrupted)

L_rpca, S_rpca, _ = rpca_admm(X_corrupted.T, max_iter=500, tol=1e-5)
L_rpca = L_rpca.T  # back to N x d
pca_rpca = PCA(n_components=k_true).fit(L_rpca)

# Compare PC subspaces: how well does each recover the true PCs?
W_true_qr, _ = np.linalg.qr(W_low_rank)

def subspace_angle(W1, W2):
    """Average cosine similarity between subspaces."""
    Q1, _ = np.linalg.qr(W1); Q2, _ = np.linalg.qr(W2)
    sv = np.linalg.svd(Q1.T @ Q2, compute_uv=False)
    return np.mean(sv**2)  # 1.0 = perfect alignment

align_pca_clean   = subspace_angle(pca_clean.components_.T, W_low_rank)
align_pca_corrupt = subspace_angle(pca_corrupt.components_.T, W_low_rank)
align_rpca        = subspace_angle(pca_rpca.components_.T, W_low_rank)

print(f"Corruption rate: {corruption_rate*100:.0f}% of entries")
print(f"Subspace alignment with true PCs (higher=better):")
print(f"  PCA on clean data:     {align_pca_clean:.4f}  (upper bound)")
print(f"  PCA on corrupted data: {align_pca_corrupt:.4f}  (degraded)")
print(f"  RPCA → PCA:            {align_rpca:.4f}  (robust recovery)")

# Visualize: singular value spectrum before/after RPCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

_, s_clean, _ = np.linalg.svd(X_clean, full_matrices=False)
_, s_corrupt, _ = np.linalg.svd(X_corrupted, full_matrices=False)
_, s_rpca_L, _ = np.linalg.svd(L_rpca, full_matrices=False)

axes[0].semilogy(range(1, 21), s_clean[:20], 'g-o', ms=5, label='Clean data', lw=1.5)
axes[0].semilogy(range(1, 21), s_corrupt[:20], 'r-o', ms=5, label='Corrupted (PCA fails)', lw=1.5)
axes[0].semilogy(range(1, 21), s_rpca_L[:20], 'b-o', ms=5, label='RPCA recovered L', lw=1.5)
axes[0].axvline(k_true + 0.5, color='gray', ls='--', lw=1.5, label=f'True rank k={k_true}')
axes[0].set_xlabel('Singular value rank'); axes[0].set_ylabel('Singular value (log)')
axes[0].set_title('Singular Value Spectrum: Corruption Ruins PCA\nRPCA Recovers Clean Structure')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Sparse component S (detected outlier locations)
axes[1].imshow(S_rpca.T != 0, cmap='Reds', aspect='auto')
axes[1].set_xlabel('Feature index (d)'); axes[1].set_ylabel('Sample index (N)')
axes[1].set_title(f'Sparse Component S: Detected Corruptions\n(True corruption: {corruption_rate*100:.0f}% of entries)')
precision = np.sum(S_rpca.T[mask] != 0) / np.sum(S_rpca.T != 0)
recall = np.sum(S_rpca.T[mask] != 0) / mask.sum()
axes[1].text(2, 10, f'Precision={precision:.2f}\nRecall={recall:.2f}', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 3: Video Background/Foreground Separation (Synthetic) ────────────────
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# Simulate a grayscale video: 50x50 pixels, 100 frames
H, W, T = 50, 50, 80  # height, width, time
pixels = H * W

# Background: static gradient (low-rank — same in every frame = rank 1)
background = np.zeros((H, W))
for i in range(H):
    for j in range(W):
        background[i, j] = 0.3 + 0.4 * i/H + 0.3 * j/W  # static gradient
background += 0.02 * np.random.randn(H, W)  # tiny bg variation

# Foreground: moving "blob" (sparse — small object in each frame)
def moving_blob(t, H, W, radius=5):
    frame = np.zeros((H, W))
    cx = int(5 + (W-10) * t / T)
    cy = int(H//2 + 10 * np.sin(2*np.pi*t/T))
    y, x = np.ogrid[:H, :W]
    mask = (x - cx)**2 + (y - cy)**2 <= radius**2
    frame[mask] = 0.8
    return frame

# Build video matrix: each column = one vectorized frame
M_video = np.zeros((pixels, T))
L_true_video = np.zeros((pixels, T))
S_true_video = np.zeros((pixels, T))

for t in range(T):
    bg = background + 0.01 * np.random.randn(H, W)
    fg = moving_blob(t, H, W)
    M_video[:, t] = bg.flatten() + fg.flatten()
    L_true_video[:, t] = bg.flatten()
    S_true_video[:, t] = fg.flatten()

# Run RPCA
print("Running RPCA on video...")
L_video, S_video, _ = rpca_admm(M_video, max_iter=200, tol=1e-5)
print(f"rank(L): {np.sum(np.linalg.svd(L_video, compute_uv=False) > 1e-3)}")

# Standard PCA for comparison
from sklearn.decomposition import PCA
n_pca = 3
pca = PCA(n_components=n_pca).fit(M_video.T)
L_pca = pca.inverse_transform(pca.transform(M_video.T)).T
S_pca = M_video - L_pca

# Visualize
frame_show = T // 2
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

def show(ax, img, title, vmin=None, vmax=None):
    ax.imshow(img.reshape(H, W), cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=9); ax.axis('off')

# Row 0: Observed frames
for col, t in enumerate([0, T//4, T//2, 3*T//4]):
    show(axes[0, col], M_video[:, t], f'Observed (t={t})', 0, 1.2)
axes[0, 0].set_ylabel('Observed M', fontsize=10)

# Row 1: RPCA background (L)
for col, t in enumerate([0, T//4, T//2, 3*T//4]):
    show(axes[1, col], L_video[:, t], f'RPCA Background (t={t})', 0, 1)
axes[1, 0].set_ylabel('RPCA Background L', fontsize=10)

# Row 2: RPCA foreground (S)
for col, t in enumerate([0, T//4, T//2, 3*T//4]):
    show(axes[2, col], np.abs(S_video[:, t]), f'RPCA Foreground (t={t})')
axes[2, 0].set_ylabel('RPCA Foreground S', fontsize=10)

plt.suptitle('Robust PCA: Video Background/Foreground Separation\n'
             'L (low-rank) = static background | S (sparse) = moving objects',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Quantitative comparison
L_err_pca = np.linalg.norm(L_pca - L_true_video, 'fro') / np.linalg.norm(L_true_video, 'fro')
L_err_rpca = np.linalg.norm(L_video - L_true_video, 'fro') / np.linalg.norm(L_true_video, 'fro')
print(f"Background recovery error:")
print(f"  Standard PCA (k={n_pca}): {L_err_pca:.3f}")
print(f"  Robust PCA:              {L_err_rpca:.3f}")